In [ ]:
namespace = "default"
ray_image = "has to be specified"
openshift_api_url = "has to be specified"
kubernetes_user_bearer_token = "has to be specified"
num_gpus = 1
script_configmap = "default"
algorithm = "sft"
pvc_name = "default"

In [ ]:
import os
import subprocess
import time

from codeflare_sdk import RayJob, ManagedClusterConfig
from kubernetes.client import V1Volume, V1VolumeMount, V1ConfigMapVolumeSource

num_gpus = int(num_gpus)

In [ ]:
result = subprocess.run(
    ["oc", "login", f"--token={kubernetes_user_bearer_token}",
     f"--server={openshift_api_url}", "--insecure-skip-tls-verify"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError(f"oc login failed: {result.stderr}")

In [ ]:
import socket

PVC_MOUNT_PATH = "/opt/app-root/src"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_S3_KEY = "models/Qwen2.5-1.5B-Instruct"
MODEL_DIR = f"{PVC_MOUNT_PATH}/model"
DATASET_DIR = f"{PVC_MOUNT_PATH}/dataset"

socket.setdefaulttimeout(10)

try:
    import s3fs
    HAS_S3FS = True
except ImportError:
    HAS_S3FS = False

s3_endpoint = os.environ.get("AWS_DEFAULT_ENDPOINT", "")
s3_access_key = os.environ.get("AWS_ACCESS_KEY_ID", "")
s3_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY", "")
s3_bucket = os.environ.get("AWS_STORAGE_BUCKET", "")
s3_prefix = os.environ.get("AWS_STORAGE_BUCKET_RAY_TRAINING_HUB_DIR", "")

model_downloaded = False

if HAS_S3FS and s3_endpoint and s3_bucket:
    try:
        endpoint_url = s3_endpoint if s3_endpoint.startswith("http") else f"https://{s3_endpoint}"
        prefix = (s3_prefix or "").strip("/")
        print(f"[notebook] S3 configured: endpoint={endpoint_url}, bucket={s3_bucket}, prefix={prefix or '<root>'}")

        fs = s3fs.S3FileSystem(
            key=s3_access_key,
            secret=s3_secret_key,
            endpoint_url=endpoint_url,
            use_ssl=endpoint_url.startswith("https"),
            config_kwargs={"signature_version": "s3v4"},
            client_kwargs={"verify": False},
        )

        model_remote = f"{s3_bucket}/{prefix}/{MODEL_S3_KEY}" if prefix else f"{s3_bucket}/{MODEL_S3_KEY}"
        os.makedirs(MODEL_DIR, exist_ok=True)
        print(f"[notebook] Downloading model from s3://{model_remote} to {MODEL_DIR}...")
        for remote_file in fs.find(model_remote):
            rel = remote_file[len(model_remote):].lstrip("/")
            if not rel or remote_file.endswith("/"):
                continue
            dst = os.path.join(MODEL_DIR, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if not os.path.exists(dst):
                fs.get(remote_file, dst)
        if os.listdir(MODEL_DIR):
            model_downloaded = True
            print(f"[notebook] Model downloaded from S3 to {MODEL_DIR}")
        else:
            print("[notebook] S3 model directory was empty")

        if algorithm == "grpo":
            dataset_remote = f"{s3_bucket}/{prefix}/grpo-dataset" if prefix else f"{s3_bucket}/grpo-dataset"
            os.makedirs(DATASET_DIR, exist_ok=True)
            print(f"[notebook] Downloading GRPO dataset from s3://{dataset_remote}...")
            for remote_file in fs.find(dataset_remote):
                rel = remote_file[len(dataset_remote):].lstrip("/")
                if not rel or remote_file.endswith("/"):
                    continue
                dst = os.path.join(DATASET_DIR, rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                if not os.path.exists(dst):
                    fs.get(remote_file, dst)
            print(f"[notebook] GRPO dataset downloaded from S3 to {DATASET_DIR}")

    except Exception as e:
        print(f"[notebook] S3 download failed: {e}")
        import traceback
        traceback.print_exc()
        print("[notebook] Will attempt HuggingFace fallback...")
else:
    if not HAS_S3FS:
        print("[notebook] S3 not available: s3fs not installed")
    else:
        print("[notebook] S3 not configured (missing endpoint or bucket env vars)")

if not model_downloaded:
    print(f"[notebook] Downloading model {MODEL_ID} from HuggingFace...")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id=MODEL_ID, local_dir=MODEL_DIR)
    print(f"[notebook] Model downloaded to {MODEL_DIR}")

    if algorithm == "grpo":
        from datasets import load_dataset, DownloadConfig

        DATASET_ID = "Agent-Ark/Toucan-1.5M"
        GRPO_SUBSET_SIZE = 200
        print(f"[notebook] Loading {GRPO_SUBSET_SIZE}-sample subset of {DATASET_ID}...")
        dataset = load_dataset(DATASET_ID, "Qwen3", data_files="Qwen3/train-00000-of-00044.parquet", split=f"train[:{GRPO_SUBSET_SIZE}]", verification_mode="no_checks")
        os.makedirs(DATASET_DIR, exist_ok=True)
        dataset.to_json(f"{DATASET_DIR}/train.jsonl")
        print(f"[notebook] Dataset subset ({len(dataset)} samples) saved to {DATASET_DIR}/train.jsonl")

print(f"[notebook] Model ready at {MODEL_DIR}")
if algorithm == "grpo":
    print(f"[notebook] Dataset ready at {DATASET_DIR}")

In [ ]:
from kubernetes.client import V1PersistentVolumeClaimVolumeSource

SCRIPT_MOUNT_PATH = "/mnt/scripts"
SHARED_DATA_MOUNT_PATH = "/mnt/shared"

env_vars = {
    "MODEL_DIR": f"{SHARED_DATA_MOUNT_PATH}/model",
    "GRPO_N_GPUS": str(num_gpus),
}
if algorithm == "grpo":
    env_vars["GRPO_DATA_PATH"] = f"{SHARED_DATA_MOUNT_PATH}/dataset/train.jsonl"
hf_token = os.environ.get("HF_TOKEN", "")
if hf_token:
    env_vars["HF_TOKEN"] = hf_token
    env_vars["HUGGING_FACE_HUB_TOKEN"] = hf_token

if algorithm == "grpo":
    head_accels = {}
    n_workers = 1
    worker_accels = {"nvidia.com/gpu": num_gpus}
    worker_cpu_req, worker_cpu_lim = 4, 8
    worker_mem_req, worker_mem_lim = 64, 96
else:
    head_accels = {"nvidia.com/gpu": num_gpus}
    n_workers = 0
    worker_accels = {}
    worker_cpu_req, worker_cpu_lim = 1, 1
    worker_mem_req, worker_mem_lim = 2, 2

cluster_config = ManagedClusterConfig(
    image=ray_image,
    head_cpu_requests=2,
    head_cpu_limits=4,
    head_memory_requests=32,
    head_memory_limits=48,
    head_accelerators=head_accels,
    num_workers=n_workers,
    worker_cpu_requests=worker_cpu_req,
    worker_cpu_limits=worker_cpu_lim,
    worker_memory_requests=worker_mem_req,
    worker_memory_limits=worker_mem_lim,
    worker_accelerators=worker_accels,
    envs=env_vars,
    volumes=[
        V1Volume(
            name="train-scripts",
            config_map=V1ConfigMapVolumeSource(name=script_configmap)
        ),
        V1Volume(
            name="shared-data",
            persistent_volume_claim=V1PersistentVolumeClaimVolumeSource(claim_name=pvc_name)
        ),
    ],
    volume_mounts=[
        V1VolumeMount(name="train-scripts", mount_path=SCRIPT_MOUNT_PATH, read_only=True),
        V1VolumeMount(name="shared-data", mount_path=SHARED_DATA_MOUNT_PATH),
    ],
)

print(f"Algorithm: {algorithm}")
print(f"Cluster config: head={head_accels}, workers={n_workers}, worker_accels={worker_accels}")
print(f"Image: {ray_image}")
print(f"Script ConfigMap '{script_configmap}' mounted at {SCRIPT_MOUNT_PATH}")
print(f"Shared PVC '{pvc_name}' mounted at {SHARED_DATA_MOUNT_PATH}")
print(f"Model available at {SHARED_DATA_MOUNT_PATH}/model")

In [ ]:
entrypoint = f"sh -c 'python {SCRIPT_MOUNT_PATH}/{algorithm}_train.py'"
print(f"Entrypoint: {entrypoint}")

In [ ]:
job = RayJob(
    job_name=f"{algorithm}-e2e-test",
    entrypoint=entrypoint,
    cluster_config=cluster_config,
    namespace=namespace,
    ttl_seconds_after_finished=60,
)

job.submit()
print(f"RayJob submitted to namespace '{namespace}'")

In [ ]:
print("Waiting for job to complete...")
final_status = None
max_polls = 360
poll_interval = 10

for i in range(max_polls):
    try:
        status, ready = job.status(print_to_console=False)
        status_name = status.name if hasattr(status, 'name') else str(status)
        print(f"[{i * poll_interval}s] Status: {status_name}")
        if status_name in ("SUCCEEDED", "COMPLETE"):
            final_status = "SUCCEEDED"
            break
        if status_name == "FAILED":
            final_status = "FAILED"
            break
    except Exception as e:
        print(f"[{i * poll_interval}s] Error checking status: {e}")
    time.sleep(poll_interval)

if final_status is None:
    final_status = "TIMEOUT"

print(f"\nFinal status: {final_status}")

In [ ]:
if final_status == "SUCCEEDED":
    print(f"{algorithm.upper()} E2E test passed")
else:
    raise RuntimeError(f"{algorithm.upper()} E2E test failed with status: {final_status}")

In [ ]:
print("Notebook execution complete")